# Stage 3 – Campaign Outcome Simulation
Owner: _name_

One qubit per selected customer, RY(θ) with θ = 2·arcsin(√p) so P(|1⟩) = p. Success = at least 3 qubits read 1.

In [1]:
import sys; sys.path.append('../src')
import numpy as np, pandas as pd
from config import *
np.random.seed(SEED)

## 1. Load the six selected customers and their Stage 1 probabilities

In [2]:
s2 = pd.read_csv(STAGE2_RESULTS)
sel = s2[s2['contact_decision'] == 1].sort_values('rank').reset_index(drop=True)
selected_ids = sel['customer_id'].tolist()
probs = sel['predicted_probability'].to_numpy()   # same Stage 1 probabilities, unchanged
pd.DataFrame({'qubit': range(6), 'customer_id': selected_ids, 'p': probs})

,qubit,customer_id,p
0,0,C0031,0.428269
1,1,C0079,0.384625
2,2,C0183,0.281598
3,3,C0318,0.249147
4,4,C0059,0.206004
5,5,C0170,0.197733


## 2. Exact answer (Poisson-binomial)

In [3]:
def prob_at_least(probs, k):
    dist = np.array([1.0])
    for p in probs:
        dist = np.convolve(dist, [1 - p, p])
    return dist[k:].sum()
exact = prob_at_least(probs, 3)
exact

np.float64(0.23573701194984345)

## 3. Quantum circuit
Qubit i ↔ selected_ids[i]. Note: Qiskit bitstrings are little-endian (qubit 0 is the rightmost character).

In [4]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(6)
for i, p in enumerate(probs):
    qc.ry(2 * np.arcsin(np.sqrt(p)), i)
qc.measure_all()

sim = AerSimulator(seed_simulator=SEED)
tqc = transpile(qc, sim, seed_transpiler=SEED)
counts = sim.run(tqc, shots=SHOTS).result().get_counts()
quantum_est = sum(c for b, c in counts.items() if b.count('1') >= 3) / SHOTS
print(qc.draw())
print('depth', tqc.depth(), 'ops', tqc.count_ops())

         ┌────────────┐ ░ ┌─┐               
   q_0: ─┤ Ry(1.4268) ├─░─┤M├───────────────
         ├────────────┤ ░ └╥┘┌─┐            
   q_1: ─┤ Ry(1.3379) ├─░──╫─┤M├────────────
         ├────────────┤ ░  ║ └╥┘┌─┐         
   q_2: ─┤ Ry(1.1188) ├─░──╫──╫─┤M├─────────
         ├────────────┤ ░  ║  ║ └╥┘┌─┐      
   q_3: ─┤ Ry(1.0452) ├─░──╫──╫──╫─┤M├──────
        ┌┴────────────┤ ░  ║  ║  ║ └╥┘┌─┐   
   q_4: ┤ Ry(0.94222) ├─░──╫──╫──╫──╫─┤M├───
        ├─────────────┤ ░  ║  ║  ║  ║ └╥┘┌─┐
   q_5: ┤ Ry(0.92162) ├─░──╫──╫──╫──╫──╫─┤M├
        └─────────────┘ ░  ║  ║  ║  ║  ║ └╥┘
meas: 6/═══════════════════╩══╩══╩══╩══╩══╩═
                           0  1  2  3  4  5 
depth 2 ops OrderedDict({'ry': 6, 'measure': 6, 'barrier': 1})


## 4. Monte Carlo check (classical)

In [5]:
rng = np.random.default_rng(SEED)
mc = ((rng.random((SHOTS, 6)) < probs).sum(axis=1) >= 3).mean()
print(f'exact={exact:.4f}  quantum={quantum_est:.4f}  monte_carlo={mc:.4f}')

exact=0.2357  quantum=0.2355  monte_carlo=0.2322


## 5. Save results

In [6]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

# Illustrative noisy run (unmitigated): 0.1% depolarizing per gate, 2% readout error
nm = NoiseModel()
nm.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ['ry', 'sx', 'x', 'rz'])
nm.add_all_qubit_readout_error(ReadoutError([[0.98, 0.02], [0.02, 0.98]]))
noisy_sim = AerSimulator(noise_model=nm, seed_simulator=SEED)
noisy_counts = noisy_sim.run(transpile(qc, noisy_sim, seed_transpiler=SEED), shots=SHOTS).result().get_counts()
noisy_est = sum(c for b, c in noisy_counts.items() if b.count('1') >= 3) / SHOTS
print(f'noisy={noisy_est:.4f}')

thetas = [float(2 * np.arcsin(np.sqrt(p))) for p in probs]
pd.DataFrame([{
    'selected_ids': ','.join(selected_ids),
    'qubit_mapping': ';'.join(f'q{i}={cid}' for i, cid in enumerate(selected_ids)),
    'input_probabilities': ';'.join(f'{p:.6f}' for p in probs),
    'ry_angles_radians': ';'.join(f'{t:.6f}' for t in thetas),
    'shots': SHOTS,
    'measurement_counts': ';'.join(f'{b}:{c}' for b, c in sorted(counts.items())),
    'quantum_estimate': quantum_est, 'exact': exact, 'monte_carlo': mc, 'noisy_estimate': noisy_est,
    'abs_error_quantum': abs(quantum_est - exact), 'abs_error_monte_carlo': abs(mc - exact),
    'abs_error_noisy': abs(noisy_est - exact), 'shot_noise_1sigma': float(np.sqrt(exact * (1 - exact) / SHOTS)),
    'logical_qubits': 6, 'depth_original': qc.depth(), 'depth_transpiled': tqc.depth(),
    'two_qubit_gates': 0, 'swaps': 0, 'backend': 'Qiskit Aer AerSimulator (ideal) + illustrative noise model',
    'seed': SEED,
}]).to_csv(STAGE3_RESULTS, index=False)

noisy=0.2509
